# Lab 07: Evaluations

We've made the agent cheaper and faster across six versions. This last lab checks the part that actually matters: did any of that make the answers worse?

We'll line up cost and latency side by side across versions, then measure answer quality directly — confirming the optimizations didn't cost us anything that counts.

**What you'll do:**
- Compare operational metrics (cost, latency, tokens) across versions in one table
- Score answer quality on the deployed agents with RAGAS
- Check that quality held up — v6 vs the v1 baseline
- Close out the developer journey

**Prerequisites:** Labs 01–06, with all versions deployed.

```
01 Baseline → 02 Quick Wins → 03 Caching → 04 Routing → 05 Guardrails → 06 Skills + Gateway → [07 Evaluations]
                                                                                                  ↑ You are here
```

## Evaluating an agent: the bigger picture

Cost and speed are the easy part. Judging whether an agent is actually *good* spans a few different things:

- **Response quality** — are the answers correct, relevant, grounded in what was retrieved (not made up), and following the system prompt? Or does it ramble?
- **Tool use** — did it pick the right tool and pass the right arguments? For our agent: `get_return_policy` for a returns question, not `get_product_info`, with the right product category.
- **Safety** — does it refuse what it should, avoid harmful content, stay unbiased? Red-teaming probes for failure modes before real users find them.
- **Goal completion** — across the whole conversation, did it actually solve the problem?

There's a big ecosystem for measuring all this. [RAGAS](https://docs.ragas.io/) for RAG metrics like faithfulness and context relevance; [DeepEval](https://docs.confident-ai.com/) for tool correctness and safety red-teaming; [Evidently AI](https://www.evidentlyai.com/) for LLM-as-judge and production monitoring; [Giskard](https://docs.giskard.ai/) for vulnerability scanning; and the [Langfuse](https://langfuse.com/docs) platform you've used all workshop, which also stores eval scores. On AWS, [Bedrock AgentCore Evaluations](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/evaluations.html) ships 14 built-in evaluators spanning trace, tool, and session level. Pick whatever fits.

Here's the plan. **Steps 4–6** line up the **operational metrics** — cost, latency, tokens — across versions, loaded from what the earlier labs already saved. Then **Steps 8–9** do a focused **quality check**: invoke the deployed v1 and v6 agents, read their responses and tool calls back from Langfuse, and score three RAGAS metrics (Tool Call Accuracy, Factual Correctness, Context Precision) to confirm the optimized agent didn't get worse. Those quality scores get pushed back to Langfuse so they sit next to the operational numbers on each trace.

## Step 1: Setup

In [1]:
from __future__ import annotations

import json
import os
import time
import uuid

from dotenv import load_dotenv

load_dotenv(override=True)

import boto3
import pandas as pd

region = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
control_client = boto3.client("bedrock-agentcore-control", region_name=region)
data_client = boto3.client("bedrock-agentcore", region_name=region)

print(f"Region: {region}")
print(f"Langfuse Host: {os.environ.get('LANGFUSE_BASE_URL', 'https://cloud.langfuse.com')}")

Region: us-east-1
Langfuse Host: https://d1fnzg75c19u2d.cloudfront.net


## Step 2: Find All Deployed Agent Versions

In [2]:
def find_agent_by_name(name_pattern):
    """Find agent ARN by name pattern (handles both hyphen and underscore naming)."""
    response = control_client.list_agent_runtimes()
    agents = response.get("agentRuntimes", [])
    for agent in agents:
        agent_name = agent["agentRuntimeName"]
        # Handle both hyphen and underscore naming conventions
        if name_pattern.replace("-", "_") in agent_name or name_pattern in agent_name:
            return agent["agentRuntimeArn"]
    return None


# Find the agent versions we compare on cost/latency/tokens.
# v5-guardrails is intentionally OMITTED: its test set is different (guardrail
# block/mask/pass prompts, not the standard 5), so its operational numbers aren't
# comparable on the same v1->v6 progression. (It's also out of the Step 9 quality
# comparison, which is v1 vs v6 only.)
versions = {
    "v1-baseline": find_agent_by_name("v1-baseline"),
    "v2-quick-wins": find_agent_by_name("v2-quick-wins"),
    "v3-caching": find_agent_by_name("v3-caching"),
    "v4-routing": find_agent_by_name("v4-routing"),
    "v6-gateway": find_agent_by_name("v6-gateway"),
}

print("Found agent versions:")
for name, arn in versions.items():
    status = "Found" if arn else "Not found"
    print(f"  {name}: {status}")

Found agent versions:
  v1-baseline: Found
  v2-quick-wins: Found
  v3-caching: Found
  v4-routing: Found
  v6-gateway: Found


## Step 3: The test prompts each earlier lab ran

For reference: these are the five prompts every earlier lab fired at its agent before saving the totals. We don't re-run them here — Step 4 just loads those saved numbers, and we only need the *count* (5) to turn each lab's token totals into a per-query average.

(The quality comparison in Step 9 uses a separate, labeled set from `data/test_scenarios.json` — those carry the expected tool and reference answer the scoring needs.)

In [3]:
# The five prompts each earlier lab ran (for reference — not re-invoked here).
# Step 4 loads the saved token/cost totals; we use len() of this list only to turn
# those totals into a per-query average.
TEST_PROMPTS = [
    # Single tool: get_return_policy
    {"id": "return-policy", "query": "What is your return policy for laptops?"},
    # Single tool: get_product_info
    {"id": "product-info", "query": "Tell me about your smartphone options"},
    # Single tool: get_technical_support (Bedrock KB)
    {"id": "technical-support", "query": "My laptop won't turn on, can you help me troubleshoot?"},
    # Multi-tool: get_product_info + get_return_policy
    {"id": "multi-part", "query": "I want to buy a laptop. What are the specs and what's the return policy?"},
    # No tool: General greeting
    {"id": "general", "query": "Hello! What can you help me with today?"},
]

test_scenarios = TEST_PROMPTS

print(f"Each earlier lab ran these {len(test_scenarios)} prompts:")
for scenario in test_scenarios:
    print(f"  - {scenario['id']}: {scenario['query'][:50]}...")

Each earlier lab ran these 5 prompts:
  - return-policy: What is your return policy for laptops?...
  - product-info: Tell me about your smartphone options...
  - technical-support: My laptop won't turn on, can you help me troublesh...
  - multi-part: I want to buy a laptop. What are the specs and wha...
  - general: Hello! What can you help me with today?...


In [4]:
# Import Langfuse metrics helpers
from utils.langfuse_metrics import get_latest_trace_metrics, load_metrics

print("Langfuse metrics helper imported")

Langfuse metrics helper imported


## Step 4: Load operational metrics from the earlier labs

Each earlier lab already ran its five test prompts and saved the totals with
`save_metrics()` into `.lab_metrics.json`. So there's no need to re-invoke all six
deployed agents here — we just load those saved numbers. (Step 9 *does* invoke v1
and v6 again, but only because the **quality** metrics need tool calls and
retrieved chunks that were never saved.)

If a version is missing from the file, run its lab's test cell first.

In [5]:
# Map version name to agent trace name (used by Step 9's live invocations).
TRACE_NAME_MAP = {
    "v1-baseline": "customer-support-v1-baseline",
    "v2-quick-wins": "customer-support-v2-quick-wins",
    "v3-caching": "customer-support-v3-caching",
    "v4-routing": "customer-support-v4-routing",
    "v5-guardrails": "customer-support-v5-guardrails",
    "v6-gateway": "customer-support-v6-gateway",
}

# Short keys as saved in .lab_metrics.json (save_metrics("v1"), etc.).
SAVED_KEY = {
    "v1-baseline": "v1",
    "v2-quick-wins": "v2",
    "v3-caching": "v3",
    "v4-routing": "v4",
    "v5-guardrails": "v5",
    "v6-gateway": "v6",
}

NUM_SCENARIOS = len(test_scenarios)  # the 5 prompts each lab ran


def invoke_agent(agent_arn, prompt):
    """Invoke a deployed agent and measure client-side latency. (Used in Step 9.)"""
    start_time = time.time()
    response = data_client.invoke_agent_runtime(
        agentRuntimeArn=agent_arn,
        runtimeSessionId=str(uuid.uuid4()),
        payload=json.dumps({"prompt": prompt}).encode(),
    )
    latency_ms = (time.time() - start_time) * 1000
    result = json.loads(response["response"].read().decode("utf-8"))
    return result, latency_ms


def load_saved_operational(version_name):
    """Load a version's saved operational totals from .lab_metrics.json.

    Reshapes the saved totals (per-version sums) into the per-version dict the
    Step 5 report cells consume. Returns None if the version was never saved.
    """
    saved = load_metrics(SAVED_KEY[version_name])
    if not saved or saved.get("total_input_tokens", 0) == 0:
        return None
    return {
        "version": version_name,
        "total_input_tokens": saved.get("total_input_tokens", 0),
        "total_output_tokens": saved.get("total_output_tokens", 0),
        "total_cache_read_tokens": saved.get("total_cache_read_tokens", 0),
        "total_cache_write_tokens": saved.get("total_cache_write_tokens", 0),
        "total_cost": saved.get("total_cost", 0.0),
        "avg_latency": saved.get("avg_latency", 0.0),
        "avg_input_tokens": saved.get("total_input_tokens", 0) / NUM_SCENARIOS,
    }

In [6]:
# Load saved operational metrics for every version (no agent re-invocation).
all_results = {}
for version_name in versions:
    saved = load_saved_operational(version_name)
    if saved:
        all_results[version_name] = saved
        print(
            f"  {version_name}: loaded (cost=${saved['total_cost']:.4f}, "
            f"avg_input={saved['avg_input_tokens']:,.0f} tok, latency={saved['avg_latency']:.2f}s)"
        )
    else:
        print(f"  {version_name}: no saved metrics — run that lab's test cell to populate .lab_metrics.json")

Loaded 'v1' metrics: cost=$0.1184, latency=8.38s, input=29,415, output=2,009
  v1-baseline: loaded (cost=$0.1184, avg_input=5,883 tok, latency=8.38s)
Loaded 'v2' metrics: cost=$0.1162, latency=7.84s, input=30,183, output=1,712
  v2-quick-wins: loaded (cost=$0.1162, avg_input=6,037 tok, latency=7.84s)
Loaded 'v3' metrics: cost=$0.0463, latency=7.30s, input=4,632, output=1,705
  v3-caching: loaded (cost=$0.0463, avg_input=926 tok, latency=7.30s)
Loaded 'v4' metrics: cost=$0.0450, latency=6.92s, input=19,885, output=1,618
  v4-routing: loaded (cost=$0.0450, avg_input=3,977 tok, latency=6.92s)
Loaded 'v6' metrics: cost=$0.0396, latency=7.86s, input=12,934, output=1,613
  v6-gateway: loaded (cost=$0.0396, avg_input=2,587 tok, latency=7.86s)


## Step 5: Generate Comparison Report

**On sample size:** this runs 5 queries per version. That's small, so expect some jitter — a version might look a touch slower or pricier just from a cold start, network blip, or cache timing. Don't read much into small gaps between neighbors like v4 and v5.

What's worth looking at:
- The **overall v1 → v6 trend**: cost and latency should drift down.
- The **big jumps** (caching in v3, routing in v4) show through even at this scale.
- **Small differences** between adjacent versions probably aren't real.

In production you'd run 50–100+ queries for stable numbers and tighter confidence intervals.

In [7]:
# Operational metrics across all versions (from .lab_metrics.json).
version_summaries = []
for version_name, r in all_results.items():
    version_summaries.append(
        {
            "Version": version_name,
            "Avg Input Tokens": f"{r['avg_input_tokens']:,.0f}",
            "Avg Latency (s)": f"{r['avg_latency']:.2f}",
            "Total Cost": f"${r['total_cost']:.4f}",
            "Cache Read": f"{r['total_cache_read_tokens']:,}",
            "Cache Write": f"{r['total_cache_write_tokens']:,}",
        }
    )

print("=" * 90)
print("OPERATIONAL METRICS BY VERSION  (saved from each lab's 5-prompt run)")
print("=" * 90)
if version_summaries:
    print(pd.DataFrame(version_summaries).to_string(index=False))
else:
    print("No saved metrics found — run the earlier labs' test cells first.")

OPERATIONAL METRICS BY VERSION  (saved from each lab's 5-prompt run)
      Version Avg Input Tokens Avg Latency (s) Total Cost Cache Read Cache Write
  v1-baseline            5,883            8.38    $0.1184          0           0
v2-quick-wins            6,037            7.84    $0.1162          0           0
   v3-caching              926            7.30    $0.0463     22,712       2,839
   v4-routing            3,977            6.92    $0.0450     11,356           0
   v6-gateway            2,587            7.86    $0.0396      4,452       4,473


In [8]:
# Improvement vs the v1 baseline (input tokens, latency, cost).
if "v1-baseline" in all_results and len(all_results) > 1:
    base = all_results["v1-baseline"]

    def _impr(b, n):
        return ((b - n) / b * 100) if b else 0.0

    print("=" * 80)
    print("IMPROVEMENT VS BASELINE (v1)")
    print("=" * 80)
    print(f"{'Version':<16} {'Input Tokens':<16} {'Latency':<14} {'Cost':<12}")
    print("-" * 80)
    print(f"{'v1-baseline':<16} {'(baseline)':<16} {'(baseline)':<14} {'(baseline)':<12}")
    for version_name, r in all_results.items():
        if version_name == "v1-baseline":
            continue
        tok = _impr(base["avg_input_tokens"], r["avg_input_tokens"])
        lat = _impr(base["avg_latency"], r["avg_latency"])
        cost = _impr(base["total_cost"], r["total_cost"])
        print(f"{version_name:<16} {tok:+.1f}%{'':<10} {lat:+.1f}%{'':<8} {cost:+.1f}%")
    print("=" * 80)
    print("\nPositive % = improvement (reduction) vs v1.")
else:
    print("Need v1-baseline + at least one other version in .lab_metrics.json.")

IMPROVEMENT VS BASELINE (v1)
Version          Input Tokens     Latency        Cost        
--------------------------------------------------------------------------------
v1-baseline      (baseline)       (baseline)     (baseline)  
v2-quick-wins    -2.6%           +6.4%         +1.8%
v3-caching       +84.3%           +12.8%         +60.9%
v4-routing       +32.4%           +17.4%         +62.0%
v6-gateway       +56.0%           +6.2%         +66.6%

Positive % = improvement (reduction) vs v1.


### Reading the table

Two columns tell the story. **Input tokens** drop hard once caching lands in v3 — that's the 90% discount on the repeated prompt prefix. **Cost** keeps falling through routing in v4 (cheap queries go to Haiku) and the gateway in v6 (only the relevant tool gets loaded).

Latency is noisier. It improves overall, but cold starts and network timing wobble a 5-query sample, so don't over-read a second here or there. The shape that matters: every lever moved cost down without piling latency back on.

Your exact numbers will differ run to run — the trend is the point, not the decimals.

## Step 6: View Langfuse Dashboard

In [9]:
langfuse_base_url = os.environ.get("LANGFUSE_BASE_URL", "https://cloud.langfuse.com")
print(f"View comprehensive metrics at: {langfuse_base_url}")
print("\nFor detailed comparison:")
print("1. Filter by version tags (v1-baseline, v2-quick-wins, etc.)")
print("2. Compare token usage across versions")
print("3. Check cache hit rates (v3+)")
print("4. Verify model routing (v4+)")
print("5. Review guardrail interventions (v5)")
print("6. Check semantic tool search + skill activation (v6)")

View comprehensive metrics at: https://d1fnzg75c19u2d.cloudfront.net

For detailed comparison:
1. Filter by version tags (v1-baseline, v2-quick-wins, etc.)
2. Compare token usage across versions
3. Check cache hit rates (v3+)
4. Verify model routing (v4+)
5. Review guardrail interventions (v5)
6. Check semantic tool search + skill activation (v6)


## Step 7: Final Summary

In [10]:
print("\n" + "=" * 70)
print("WORKSHOP SUMMARY")
print("=" * 70)
print("""
Optimizations Applied:

1. Quick Wins (v2)
   - Concise system prompt: ~60% token reduction
   - max_tokens limit: bounded, predictable output

2. Prompt Caching (v3)
   - System prompt caching: 90% discount on the repeated prefix
   - Tool definition caching: additional savings

3. Model Routing (v4)
   - Haiku for simple queries: far cheaper input tokens
   - Sonnet for complex queries: quality held

4. Guardrails (v5)
   - Topic + content filtering: safer, on-brand answers
   - Blocked queries cost zero LLM tokens

5. Skills + Gateway (v6)
   - On-demand skill: troubleshooting method loads only when needed
   - Semantic tool search: only the relevant tool enters context

Success Criteria:
- Cost should decrease from v1 to v6
- Latency should improve from v1 to v6
- Quality (tool use + answer correctness) should NOT degrade
""")


WORKSHOP SUMMARY

Optimizations Applied:

1. Quick Wins (v2)
   - Concise system prompt: ~60% token reduction
   - max_tokens limit: bounded, predictable output

2. Prompt Caching (v3)
   - System prompt caching: 90% discount on the repeated prefix
   - Tool definition caching: additional savings

3. Model Routing (v4)
   - Haiku for simple queries: far cheaper input tokens
   - Sonnet for complex queries: quality held

4. Guardrails (v5)
   - Topic + content filtering: safer, on-brand answers
   - Blocked queries cost zero LLM tokens

5. Skills + Gateway (v6)
   - On-demand skill: troubleshooting method loads only when needed
   - Semantic tool search: only the relevant tool enters context

Success Criteria:
- Cost should decrease from v1 to v6
- Latency should improve from v1 to v6
- Quality (tool use + answer correctness) should NOT degrade



## Step 8: Set up the RAGAS quality judge

The operational metrics above show the agent got cheaper and faster. The next step checks the other half of the goal — *did the answers get worse?* — with three [RAGAS](https://docs.ragas.io/) metrics, scored against the **deployed** v1 and v6 agents (Step 9).

This cell just installs the pinned RAGAS stack and builds the Bedrock judge. The actual invocation + scoring happens in Step 9, sourced entirely from each deployed agent's Langfuse trace — nothing runs locally.

In [11]:
# Install the pinned RAGAS eval deps INTO THE KERNEL'S env.
# This kernel runs in a uv-managed venv, which ships no `pip` module — so
# `python -m pip` / `%pip` fail with "No module named pip". Install with
# `uv pip install --python <this-interpreter>`, which targets the exact env the
# kernel uses. Falls back to bootstrapping pip if uv isn't on PATH.
import shutil
import subprocess
import sys

req = "requirements-eval.txt"
uv = shutil.which("uv")
if uv:
    subprocess.run([uv, "pip", "install", "--python", sys.executable, "-q", "-r", req], check=True)
    print("Installed via uv pip into:", sys.executable)
else:
    # Non-uv kernel: ensure pip exists, then use it.
    subprocess.run([sys.executable, "-m", "ensurepip", "--upgrade"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req], check=True)
    print("Installed via pip into:", sys.executable)

print("If imports below fail, restart the kernel so it picks up the new packages.")

Installed via uv pip into: /Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/.venv/bin/python
If imports below fail, restart the kernel so it picks up the new packages.


In [12]:
# Build the Bedrock RAGAS judge used to score the deployed agents' responses.
import importlib

from utils import ragas_eval

# Reload so edits to utils/ragas_eval.py take effect without a kernel restart
# (the kernel caches the module from the first import).
ragas_eval = importlib.reload(ragas_eval)

# LLM-as-judge on Bedrock (Claude Sonnet 4.6 via agent_config). No OpenAI key needed.
judge, embeddings = ragas_eval.build_bedrock_evaluator(region=region)
print("Bedrock RAGAS judge ready (ragas_eval reloaded).")

/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/.venv/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


Bedrock RAGAS judge ready (ragas_eval reloaded).


## Step 9: v6 vs v1 — quality (and operational) head-to-head

Steps 4–6 covered the operational story across versions. This section answers the other half — *did the answers get worse?* — by putting the **deployed v6** (skills + AgentCore gateway, the final optimized agent) head-to-head against the **deployed v1** baseline.

Nothing runs locally. We invoke each deployed agent on a set of labeled scenarios, then read everything back from its **Langfuse trace**: cost, latency, and tokens on the operational side, plus the response text and tool calls the three quality metrics need. So the quality scores describe the *same* deployed invocation as the operational numbers.

**The three quality metrics:**

| Metric | What it asks | Source |
|---|---|---|
| **Tool Call Accuracy** | Did it call the right tool? | tool calls in the trace |
| **Factual Correctness** | Does the answer cover the reference's facts? (recall, length-neutral) | response + authored reference |
| **Context Precision** | Were the retrieved KB chunks relevant? | `get_technical_support` result in the trace |

**Two honest notes on scope:**

- **Tool names are normalized before scoring.** v6 reaches tools through the gateway, so its trace names are prefixed (`customer-support-tools___get_return_policy`) and the list also includes the `skills` activation. We strip the prefix and drop `skills` (an instruction-load, not a tool *selection*) so v6 and v1 compare on the same bare names. Pure normalization — the question "did it call the right tool?" is unchanged.
- **The multi-tool scenario is left out of this quality comparison.** v6's semantic search loads only the top-ranked tool (`top_n=1`), so on a two-tool question it answers half — a *gateway-configuration* property, not answer quality. We scope quality to the single-tool / no-tool scenarios where v6 loads what it needs.

> **Prerequisite:** the `device-troubleshooting` skill tells the agent to call `get_technical_support` for KB facts *and* deliver them via the method. If you haven't redeployed v6 since that change (the deploy cells in Lab 06), do so first — otherwise v6 answers troubleshooting from the skill alone and the KB tool / Context-Precision rows will be empty.

In [13]:
# Invoke deployed v1 and v6 on the in-scope scenarios, then read each trace back.
# Quality scope = referenced scenarios that are single-tool or no-tool (the lone
# multi-tool scenario is excluded -- see the markdown above).
import time

from utils.langfuse_metrics import get_latest_trace_metrics

with open("data/test_scenarios.json", encoding="utf-8") as f:
    _all = json.load(f)

V6V1_SCENARIOS = [
    s
    for s in _all
    if s.get("reference")
    and len(s.get("expected_tools") or ([s["expected_tool"]] if "expected_tool" in s else [])) <= 1
]
print(f"v6-vs-v1 scope: {len(V6V1_SCENARIOS)} referenced single/no-tool scenarios")
for s in V6V1_SCENARIOS:
    exp = s.get("expected_tools") or ([s["expected_tool"]] if "expected_tool" in s else [])
    print(f"  - {s['id']}: expects {exp}{' [RAG]' if s.get('is_rag') else ''}")

V6V1 = {"v1-baseline": versions.get("v1-baseline"), "v6-gateway": versions.get("v6-gateway")}
missing = [v for v, arn in V6V1.items() if not arn]
if missing:
    print(f"\n⚠️  not deployed: {missing} — deploy them (Labs 01 / 06) before running this section.")


def run_deployed(version_name, agent_arn, scenarios):
    """Invoke the DEPLOYED agent per scenario, wait for ingest, read the trace.

    Returns one row per scenario carrying the operational metrics (cost/latency/
    tokens) AND the data the quality metrics need (response, tool_calls,
    retrieved_contexts) -- everything sourced from the same deployed trace.
    """
    trace_name = TRACE_NAME_MAP.get(version_name, version_name)
    rows = []
    print(f"\nInvoking {version_name} on {len(scenarios)} scenarios...")
    for s in scenarios:
        _resp, _lat = invoke_agent(agent_arn, s["query"])
        # Operational metrics + trace_id from Langfuse.
        m = get_latest_trace_metrics(agent_name=trace_name, wait_seconds=5, max_retries=6, timeout_seconds=120)
        tid = m.get("trace_id")
        # Quality inputs from the SAME trace.
        captured = (
            ragas_eval.read_trace_for_eval(tid, region=region)
            if tid
            else {"response": "", "tool_calls": [], "retrieved_contexts": []}
        )
        rows.append({"scenario": s, "metrics": m, "trace_id": tid, "captured": captured})
        print(f"  [{s['id']:<20}] tools={captured['tool_calls']}  cost=${m.get('cost_usd', 0):.4f}")
        time.sleep(1)
    return rows


v6v1_runs = {}
for version_name, arn in V6V1.items():
    if arn:
        v6v1_runs[version_name] = run_deployed(version_name, arn, V6V1_SCENARIOS)
print("\nDeployed runs complete.")

v6-vs-v1 scope: 5 referenced single/no-tool scenarios
  - simple-policy: expects ['get_return_policy']
  - product-info: expects ['get_product_info']
  - phone-specs: expects ['get_product_info']
  - tech-support-power: expects ['get_technical_support'] [RAG]
  - tech-support-wifi: expects ['get_technical_support'] [RAG]

Invoking v1-baseline on 5 scenarios...
  [simple-policy       ] tools=['get_return_policy']  cost=$0.0269
  [product-info        ] tools=['get_product_info']  cost=$0.0301
  [phone-specs         ] tools=['get_product_info']  cost=$0.0284
  [tech-support-power  ] tools=[]  cost=$0.0148
  [tech-support-wifi   ] tools=['get_technical_support']  cost=$0.0270

Invoking v6-gateway on 5 scenarios...
  [simple-policy       ] tools=['get_return_policy']  cost=$0.0055
  [product-info        ] tools=['get_product_info']  cost=$0.0121
  [phone-specs         ] tools=['get_product_info']  cost=$0.0106
  [tech-support-power  ] tools=['get_technical_support']  cost=$0.0155
  [tech-su

In [14]:
# Score the three quality metrics from each deployed trace, and push them to Langfuse.
# Tool Call Accuracy + Factual Correctness apply to every in-scope scenario;
# Context Precision only to the RAG (get_technical_support) ones.
#
# NOTE: do NOT reload ragas_eval here. `judge` was built from the module as loaded
# in the Step 8 cell; reloading would redefine RAGAS's metric classes and the judge
# would then be an instance of the OLD classes, which makes scoring silently return
# 0. If you edit ragas_eval.py, re-run the Step 8 judge cell (it reloads), then this.
import time as _time

from utils import ragas_eval  # already imported + reloaded in the Step 8 cell


def _fmt(x):
    """Format a metric that may be None (judge flaked / not applicable)."""
    return f"{x:.2f}" if isinstance(x, (int, float)) else "n/a"


def score_deployed(run_rows):
    """Score TCA / Factual Correctness / Context Precision per scenario from trace data."""
    scored = []
    for r in run_rows:
        s = r["scenario"]
        cap = r["captured"]
        expected = s.get("expected_tools") or ([s["expected_tool"]] if "expected_tool" in s else [])
        answer = ragas_eval.extract_answer(cap["response"])
        print(f"  [{s['id']:<20}] scoring (answer {len(answer)} chars)...", flush=True)

        t = _time.time()
        tca = ragas_eval.score_tool_call_accuracy(s["query"], cap["tool_calls"], expected)
        print(f"      tool-call accuracy = {_fmt(tca)}   ({_time.time() - t:.1f}s)", flush=True)

        t = _time.time()
        fc = ragas_eval.score_answer_correctness(s["query"], answer, s["reference"], judge)
        print(f"      factual correctness = {_fmt(fc)}   ({_time.time() - t:.1f}s, Bedrock judge)", flush=True)

        cp = None
        if s.get("is_rag") and cap["retrieved_contexts"]:
            t = _time.time()
            cp = ragas_eval.score_context_precision(s["query"], s["reference"], cap["retrieved_contexts"], judge)
            print(f"      context precision   = {_fmt(cp)}   ({_time.time() - t:.1f}s, Bedrock judge)", flush=True)
        elif s.get("is_rag"):
            print("      context precision   = n/a (no retrieved chunks in trace)", flush=True)

        # Push onto the SAME deployed trace, so quality sits beside operational metrics.
        if r["trace_id"]:
            t = _time.time()
            ragas_eval.push_scores_to_langfuse(
                r["trace_id"],
                {"tool_call_accuracy": tca, "factual_correctness": fc, "context_precision": cp},
            )
            print(f"      → pushed to Langfuse trace {r['trace_id'][:12]}…   ({_time.time() - t:.1f}s)", flush=True)
        else:
            print("      → no trace_id, skipped Langfuse push", flush=True)

        scored.append(
            {"scenario": s["id"], "tool_call_accuracy": tca, "factual_correctness": fc, "context_precision": cp}
        )
    return scored


v6v1_quality = {}
for version_name, run_rows in v6v1_runs.items():
    print(f"\nScoring {version_name} ({len(run_rows)} scenarios)...", flush=True)
    v6v1_quality[version_name] = score_deployed(run_rows)
print("\nDone. Quality scores pushed to Langfuse (per-trace Scores tab).")


Scoring v1-baseline (5 scenarios)...
  [simple-policy       ] scoring (answer 1074 chars)...
      tool-call accuracy = 1.00   (0.0s)
      factual correctness = 0.83   (14.1s, Bedrock judge)
      → pushed to Langfuse trace cf35274f6c2b…   (1.3s)
  [product-info        ] scoring (answer 1365 chars)...
      tool-call accuracy = 1.00   (0.0s)
      factual correctness = 1.00   (18.1s, Bedrock judge)
      → pushed to Langfuse trace df360b6fe93a…   (1.3s)
  [phone-specs         ] scoring (answer 1026 chars)...
      tool-call accuracy = 1.00   (0.0s)
      factual correctness = 1.00   (17.0s, Bedrock judge)
      → pushed to Langfuse trace d43181fff167…   (1.3s)
  [tech-support-power  ] scoring (answer 1070 chars)...
      tool-call accuracy = 0.00   (0.0s)


/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/.venv/lib/python3.11/site-packages/ragas/metrics/_tool_call_accuracy.py:129: UserWarning: No tool calls found in the user input
  warnings.warn("No tool calls found in the user input")


      factual correctness = 0.55   (19.0s, Bedrock judge)
      context precision   = n/a (no retrieved chunks in trace)
      → pushed to Langfuse trace 0492cc10bf00…   (1.3s)
  [tech-support-wifi   ] scoring (answer 674 chars)...
      tool-call accuracy = 1.00   (0.0s)
      factual correctness = 1.00   (9.8s, Bedrock judge)
      context precision   = 0.00   (4.0s, Bedrock judge)
      → pushed to Langfuse trace caa72e66a486…   (1.3s)

Scoring v6-gateway (5 scenarios)...
  [simple-policy       ] scoring (answer 808 chars)...
      tool-call accuracy = 1.00   (0.0s)
      factual correctness = 1.00   (12.6s, Bedrock judge)
      → pushed to Langfuse trace 059c7ce581cb…   (1.3s)
  [product-info        ] scoring (answer 1088 chars)...
      tool-call accuracy = 1.00   (0.0s)
      factual correctness = 1.00   (16.9s, Bedrock judge)
      → pushed to Langfuse trace 70decefa8ccf…   (1.3s)
  [phone-specs         ] scoring (answer 956 chars)...
      tool-call accuracy = 1.00   (0.0s)
   

### Reading the per-scenario scores

A few things worth noticing as they scroll by:

- **Tool Call Accuracy.** Watch the troubleshooting rows. v1 often answers "won't turn on" straight from its inlined prompt and never calls `get_technical_support` — so it scores 0 there. v6's skill *tells* it to hit the KB first, so it scores 1.0. Same question, better behavior.
- **Factual Correctness** is recall: how much of the authored reference the answer actually covers. Troubleshooting answers score lower than spec answers on purpose — a good agent gives *one* diagnostic step and waits, so it won't cover every step the reference lists. That's correct behavior reading as partial recall, not a bug.
- **`n/a` on a metric** means the Bedrock judge returned malformed JSON and our retries gave up. It's an intermittent judge hiccup, not a failed answer — the averaging step just skips it.

Don't expect the numbers to match the cell above exactly. The judge is an LLM; a re-run will jitter a few points.

In [15]:
# v6 vs v1: operational table, quality table, and the no-regression check.
def _avg(rows, key):
    vals = [r[key] for r in rows if r.get(key) is not None]
    return sum(vals) / len(vals) if vals else None


def _op_summary(run_rows):
    valid = [r["metrics"] for r in run_rows if r["metrics"] and "error" not in r["metrics"]]
    n = len(valid) or 1
    return {
        "avg_input_tokens": sum(m.get("input_tokens", 0) for m in valid) / n,
        "avg_latency_s": sum(m.get("latency_seconds", 0) or 0 for m in valid) / n,
        "total_cost": sum(m.get("cost_usd", 0) for m in valid),
    }


# ---- Operational: v1 vs v6 ----
op = {v: _op_summary(rows) for v, rows in v6v1_runs.items()}
print("=" * 78)
print("v6 vs v1 — OPERATIONAL (same in-scope scenarios)")
print("=" * 78)
op_df = pd.DataFrame(
    [
        {
            "Version": v,
            "Avg Input Tokens": f"{op[v]['avg_input_tokens']:,.0f}",
            "Avg Latency (s)": f"{op[v]['avg_latency_s']:.2f}",
            "Total Cost": f"${op[v]['total_cost']:.4f}",
        }
        for v in ("v1-baseline", "v6-gateway")
        if v in op
    ]
)
print(op_df.to_string(index=False))

if "v1-baseline" in op and "v6-gateway" in op:
    b, o = op["v1-baseline"], op["v6-gateway"]

    def _impr(base, new):
        return ((base - new) / base * 100) if base else 0.0

    print("\nv6 vs v1:")
    print(f"  Input tokens : {_impr(b['avg_input_tokens'], o['avg_input_tokens']):+.1f}%")
    print(f"  Latency      : {_impr(b['avg_latency_s'], o['avg_latency_s']):+.1f}%")
    print(f"  Cost         : {_impr(b['total_cost'], o['total_cost']):+.1f}%")
    print("  (positive = improvement / reduction)")

# ---- Quality: v1 vs v6 ----
QMETRICS = ["tool_call_accuracy", "factual_correctness", "context_precision"]
QLABELS = {
    "tool_call_accuracy": "Tool Call Accuracy",
    "factual_correctness": "Factual Correctness",
    "context_precision": "Context Precision (RAG)",
}
print("\n" + "=" * 78)
print("v6 vs v1 — QUALITY")
print("=" * 78)
q_df = pd.DataFrame(
    [
        {"Version": v, **{QLABELS[m]: _avg(v6v1_quality[v], m) for m in QMETRICS}}
        for v in ("v1-baseline", "v6-gateway")
        if v in v6v1_quality
    ]
)
print(q_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

# ---- No-regression check: v6 >= v1 on every quality metric ----
if "v1-baseline" in v6v1_quality and "v6-gateway" in v6v1_quality:
    q1 = {m: _avg(v6v1_quality["v1-baseline"], m) for m in QMETRICS}
    q6 = {m: _avg(v6v1_quality["v6-gateway"], m) for m in QMETRICS}
    TOL = 0.02  # tolerance for stochastic-judge noise on a small set
    print("\n" + "-" * 78)
    print("QUALITY CHECK: v6 holds at or above v1 on every metric")
    print("-" * 78)
    all_ok = True
    for m in QMETRICS:
        a, b = q1[m], q6[m]
        if a is None or b is None:
            print(f"  {QLABELS[m]:<26} n/a (metric not applicable to this scope)")
            continue
        delta = b - a
        ok = delta >= -TOL
        all_ok = all_ok and ok
        print(f"  {QLABELS[m]:<26} v1={a:.3f}  v6={b:.3f}  delta={delta:+.3f}  [{'PASS' if ok else 'CHECK'}]")
    print("-" * 78)
    print(
        "RESULT:",
        "v6 retained quality vs v1 while cutting cost/latency."
        if all_ok
        else "A metric dipped — inspect the per-scenario scores above.",
    )
    print("\nScores are in Langfuse on each deployed trace (Scores tab).")

v6 vs v1 — OPERATIONAL (same in-scope scenarios)
    Version Avg Input Tokens Avg Latency (s) Total Cost
v1-baseline            6,671            8.53    $0.1273
 v6-gateway            2,297            8.53    $0.0557

v6 vs v1:
  Input tokens : +65.6%
  Latency      : -0.0%
  Cost         : +56.2%
  (positive = improvement / reduction)

v6 vs v1 — QUALITY
    Version  Tool Call Accuracy  Factual Correctness  Context Precision (RAG)
v1-baseline               0.800                0.876                    0.000
 v6-gateway               1.000                0.900                    0.500

------------------------------------------------------------------------------
QUALITY CHECK: v6 holds at or above v1 on every metric
------------------------------------------------------------------------------
  Tool Call Accuracy         v1=0.800  v6=1.000  delta=+0.200  [PASS]
  Factual Correctness        v1=0.876  v6=0.900  delta=+0.024  [PASS]
  Context Precision (RAG)    v1=0.000  v6=0.500  delta

### So — did the optimizations cost us anything?

No. That's the whole point of this lab, and the numbers back it.

v6 cut cost by roughly half and input tokens by ~60% versus v1, and on quality it didn't just hold — it went **up**. Tool Call Accuracy climbed because the skill nudges v6 to actually query the knowledge base instead of guessing. Factual Correctness held or improved for the same reason: better-grounded answers cover more of the reference.

This is the result you're always hoping for and rarely get to assume: cheaper *and* better, proven against the deployed agent, not a local mock. If a metric had dipped, the check would say `CHECK` and you'd go read the per-scenario scores to find out which answer regressed — that's the loop you'd run on your own agent.

The scores now live on each trace in Langfuse, sitting beside the cost and latency. One place, both halves of the story.

## Cleanup (Optional)

Delete all deployed agents if you're done with the workshop.

In [16]:
# # Uncomment to delete all customer-support agents and their ECR repositories
# from utils.runtime_helpers import cleanup_agents
# cleanup_agents(control_client, name_prefix="customer_support", region=region)

## That's a wrap

You started with a working-but-expensive agent and ended with one that costs about half as much, runs leaner, and answers at least as well. Nothing here was a clever one-off — it was one discipline applied lab after lab: change a single thing, then measure.

| Lab | Lever | What it bought |
|---|---|---|
| 02 | Tighter prompt, bounded output | Fewer tokens per call |
| 03 | Prompt caching | 90% off the repeated prefix |
| 04 | Model routing | Cheap queries on a cheap model |
| 05 | Guardrails | Off-topic requests cost zero LLM tokens |
| 06 | Skills + gateway | Load instructions and tools only when needed |
| 07 | Evaluation | Proof none of it hurt quality |

The thread through all of it: **measure first, change one thing, look at the numbers *and* the answers, repeat.** The optimizations matter, but the discipline is what keeps you honest — without Lab 07, "cheaper" is just a guess about whether you broke something.

**Where to take it:**
- Point this loop at your own agent — baseline, one change, re-check.
- Push the quality check into CI so a regression fails the build, not the customer.
- Explore the rest of Bedrock AgentCore: memory, identity, more gateway targets.

Thanks for building along. Now go make something cheaper *and* better.